## Evualuacion de los modelos

In [1]:
# Required Libraries
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# Import your utilities and constants
from utils.model_utils import save_model, load_model
from constants import (
    X_train, X_train_scaled,
    y_train, y_train_scaled,
)

c:\Projects\Espol\machine-learning-project\machine_learning_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Por si el ganador fue un modelo que utilizó los datos escalados

In [ ]:
def is_scaled_model(model_name: str) -> bool:
    """Determine if the model uses scaled data based on its name."""
    scaled_models = ['rf4', 'rf5', 'rf6', 'lr2']
    return any(name in model_name for name in scaled_models)

### Optimizador si ganó el RF

1. Parámetros Numéricos:

   n_estimators (100-300):
   - Mínimo de 100 árboles para garantizar estabilidad del ensamble
   - 300 como máximo porque:
     a) La mejora en rendimiento tiende a estabilizarse después de 300 árboles
     b) Balance entre tiempo de computación y rendimiento
     c) Suficiente para capturar patrones complejos en datos de fraude

   max_depth (10-13):
   - Rango optimizado basado en la naturaleza del problema de fraude
   - Mínimo 10 para permitir capturar relaciones complejas
   - Máximo 13 para:
     a) Evitar sobreajuste
     b) Mantener interpretabilidad del modelo
     c) Reducir tiempo de entrenamiento

   min_samples_split (2-10):
   - Mínimo 2 por ser el valor más pequeño posible para una división
   - Máximo 10 porque:
     a) Suficiente para asegurar divisiones significativas
     b) Ayuda a prevenir sobreajuste
     c) Apropiado para datos de fraude donde algunos patrones pueden ser raros

   min_samples_leaf (1-5):
   - Rango reducido pero efectivo
   - Mínimo 1 para permitir casos específicos de fraude
   - Máximo 5 para:
     a) Asegurar predicciones robustas
     b) Evitar nodos muy pequeños
     c) Mantener generalización

2. Parámetros Categóricos:

   max_features ['sqrt', 'log2']:
   - Se mantienen ambas opciones porque:
     a) 'sqrt': 
        * Funciona bien cuando hay muchas características relevantes
        * Reduce varianza entre árboles
        * Estándar en problemas de clasificación
     b) 'log2':
        * Puede ser mejor cuando hay pocas características importantes
        * Puede capturar relaciones más específicas
        * Útil en datos de fraude donde algunas características pueden ser más determinantes

   class_weight ['balanced', 'balanced_subsample']:
   - Crucial mantener ambas opciones por el desbalance en datos de fraude
   - 'balanced':
     a) Ajusta pesos inversamente proporcional a frecuencias de clase
     b) Consistente a través de todo el modelo
     c) Bueno para desbalance general
   - 'balanced_subsample':
     a) Recalcula pesos en cada subconjunto
     b) Puede adaptarse mejor a variaciones locales
     c) Útil si el desbalance varía en diferentes segmentos de datos

3. Decisiones de Evaluación:

   Uso de cross_val_score con cv=5:
   - 5 pliegues proporcionan:
     a) Evaluación robusta del rendimiento
     b) Balance entre tiempo de computación y confiabilidad
     c) Suficiente datos en cada pliegue para evaluar casos de fraude

   Métrica balanced_accuracy:
   - Elegida porque:
     a) Requerida específicamente por el proyecto
     b) Mejor que accuracy regular para datos desbalanceados
     c) Considera igualmente importante ambas clases
     d) Crucial en detección de fraude donde falsos negativos son costosos

4. Beneficios Generales de esta Configuración:

   - Eficiencia Computacional:
     * Rangos optimizados reducen tiempo de búsqueda
     * Mantiene opciones críticas para rendimiento
     * Permite exploración suficiente del espacio de hiperparámetros

   - Calidad del Modelo:
     * Balance entre complejidad y generalización
     * Adaptabilidad a características del problema de fraude
     * Robustez en la evaluación

   - Práctica en Producción:
     * Configuración realista para entornos productivos
     * Tiempos de entrenamiento manejables
     * Facilita mantenimiento y reentrenamiento

In [ ]:
def objective_rf(trial, X, y):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500), 
        'max_depth': trial.suggest_int('max_depth', 10, 20),        
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10), 
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5), 
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced', 'balanced_subsample'])  # Mantenido ambas opciones
    }
    
    model = RandomForestClassifier(**params, random_state=42)
    scores = cross_val_score(model, X, y, cv=5, scoring='balanced_accuracy')
    return scores.mean()

### Optimizador si ganó el LogR

1. Parámetros de Regularización:

   C (1e-5 a 1e5):
   - Rango amplio logarítmico para explorar diferentes niveles de regularización
   - Valores pequeños (1e-5) para:
     a) Regularización fuerte
     b) Prevenir sobreajuste
     c) Modelos más simples y generalizables
   - Valores grandes (1e5) para:
     a) Regularización débil
     b) Permitir ajuste más preciso a los datos
     c) Capturar patrones más específicos de fraude
   - Escala logarítmica porque:
     a) Permite explorar órdenes de magnitud
     b) Más eficiente que búsqueda lineal
     c) Común en optimización de hiperparámetros

2. Parámetros de Convergencia:

   max_iter (100-1000):
   - Rango amplio para asegurar convergencia
   - Mínimo 100 para:
     a) Dar tiempo suficiente para convergencia básica
     b) Manejar datos complejos
   - Máximo 1000 para:
     a) Permitir convergencia en casos difíciles
     b) Evitar tiempo excesivo de entrenamiento
     c) Manejar posibles dificultades de convergencia con datos desbalanceados

3. Manejo de Clases Desbalanceadas:

   class_weight ['balanced', None]:
   - Dos opciones para flexibilidad:
     a) 'balanced': 
        * Ajusta pesos inversamente proporcional a frecuencias
        * Crucial para datos desbalanceados de fraude
        * Ayuda a no ignorar la clase minoritaria
     b) None:
        * Sin ajuste de pesos
        * Útil si el desbalance no afecta significativamente
        * Permite que Optuna determine si se necesita balanceo

4. Algoritmos de Optimización:

   solver ['lbfgs', 'liblinear', 'saga']:
   - Múltiples opciones por sus diferentes ventajas:
     a) 'lbfgs':
        * Optimizador por defecto
        * Bueno para la mayoría de problemas
        * Eficiente en memoria
     b) 'liblinear':
        * Eficiente para datasets pequeños/medianos
        * Bueno con L1 regularization
        * Maneja bien problemas mal condicionados
     c) 'saga':
        * Versión mejorada de 'sag'
        * Eficiente para datasets grandes
        * Compatible con L1 y L2

5. Regularización:

   penalty ['l1', 'l2']:
   - Ambas opciones incluidas porque:
     a) L1 (Lasso):
        * Puede realizar selección de características
        * Útil si hay características irrelevantes
        * Produce modelos más sparse
     b) L2 (Ridge):
        * Más estable
        * Previene sobreajuste
        * Maneja mejor colinealidad

6. Manejo de Compatibilidad:

   Código de compatibilidad solver-penalty:
   ```python
   if params['solver'] == 'lbfgs' and params['penalty'] == 'l1':
       params['penalty'] = 'l2'
   ```
   - Necesario porque:
     a) 'lbfgs' no es compatible con regularización L1
     b) Evita errores durante la optimización
     c) Mantiene la búsqueda de hiperparámetros válida

7. Evaluación:

   cross_val_score con cv=5:
   - Evaluación robusta mediante:
     a) Validación cruzada de 5 pliegues
     b) Balanced accuracy como métrica
     c) Consistente con requerimientos del proyecto

8. Beneficios Generales:

   - Exploración Completa:
     * Cubre rangos significativos de hiperparámetros
     * Permite diferentes estrategias de regularización
     * Flexible en manejo de desbalance

   - Robustez:
     * Maneja diferentes escenarios de datos
     * Previene configuraciones inválidas
     * Evaluación confiable del rendimiento

   - Eficiencia:
     * Búsqueda inteligente en espacio de hiperparámetros
     * Balance entre exploración y tiempo computacional
     * Práctica para entornos reales

In [2]:
def objective_lr(trial, X, y):
    """Optimization objective function for LogisticRegression."""
    params = {
        'C': trial.suggest_loguniform('C', 1e-5, 1e5),
        'max_iter': trial.suggest_int('max_iter', 100, 1000),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced', None]),
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'saga']),
        'penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
    }
    
    if params['solver'] == 'lbfgs' and params['penalty'] == 'l1':
        params['penalty'] = 'l2'
    
    model = LogisticRegression(**params, random_state=42)
    scores = cross_val_score(model, X, y, cv=5, scoring='balanced_accuracy')
    return scores.mean()

Cargamos el modelo a optimizar

In [ ]:
# Load the best model using your utility function
best_model = load_model('best_model')

# Print model information
print("Best model information:")
print(f"Model type: {type(best_model).__name__}")

Asignarles los datasets correctos al modelo electo

In [3]:
# Get model name and determine data type
model_name = getattr(best_model, 'model_name', 'unknown')
use_scaled_data = is_scaled_model(model_name)

# Select appropriate data
if use_scaled_data:
    print("Using scaled data for optimization")
    X = X_train_scaled
    y = y_train_scaled.values.ravel()  # Convert to 1D array
else:
    print("Using original (unscaled) data for optimization")
    X = X_train
    y = y_train.values.ravel()  # Convert to 1D array

Best model information:
Model type: RandomForestClassifier
Using original (unscaled) data for optimization


Seleccionamos la funcion objetivo apropiada

In [ ]:
if isinstance(best_model, RandomForestClassifier):
    objective = lambda trial: objective_rf(trial, X, y)
    print("Optimizing RandomForest model")
else:
    objective = lambda trial: objective_lr(trial, X, y)
    print("Optimizing LogisticRegression model")

Creamos y ejecutamos el estudio

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

Aquí se irán imprimiendo los resultados de las iteraciones

In [4]:
print("\nBest trial:")
print("  Value: ", study.best_trial.value)
print("  Params: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-04-28 23:33:49,693] A new study created in memory with name: no-name-4e8e9093-3902-483c-8f33-49b8ade50d79


Optimizing RandomForest model


[I 2025-04-28 23:41:10,987] Trial 0 finished with value: 0.8927262063933261 and parameters: {'n_estimators': 484, 'max_depth': 15, 'min_samples_split': 15, 'min_samples_leaf': 10, 'max_features': 'log2', 'class_weight': 'balanced_subsample'}. Best is trial 0 with value: 0.8927262063933261.
[I 2025-04-28 23:49:33,129] Trial 1 finished with value: 0.8877264460913681 and parameters: {'n_estimators': 729, 'max_depth': 10, 'min_samples_split': 9, 'min_samples_leaf': 5, 'max_features': 'log2', 'class_weight': 'balanced_subsample'}. Best is trial 0 with value: 0.8927262063933261.
[I 2025-04-28 23:54:41,356] Trial 2 finished with value: 0.8705910849811691 and parameters: {'n_estimators': 662, 'max_depth': 6, 'min_samples_split': 19, 'min_samples_leaf': 9, 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'}. Best is trial 0 with value: 0.8927262063933261.
[W 2025-04-28 23:57:45,996] Trial 3 failed with parameters: {'n_estimators': 861, 'max_depth': 12, 'min_samples_split': 12, 'min_sa

KeyboardInterrupt: 

Creamos nuevamente el modelo con la mejor combinacion de hiperparámetros

In [ ]:
if isinstance(best_model, RandomForestClassifier):
    optimized_model = RandomForestClassifier(**study.best_trial.params, random_state=42)
else:
    optimized_model = LogisticRegression(**study.best_trial.params, random_state=42)

Entrenamos el modelo optimizado

In [ ]:
optimized_model.fit(X, y)

Guardamos el modelo

In [ ]:
model_name = 'optimized_rf' if isinstance(best_model, RandomForestClassifier) else 'optimized_lr'
model_name = f"{model_name}_scaled" if use_scaled_data else model_name
save_model(optimized_model, model_name)